In [3]:
from pathlib import Path
while not (Path.cwd() / '.git').exists():
    %cd ..

/home/matthew/study/grab-voc-triage


Hypothesis:
- DRIVER_OPERATIONS: Does this evaluate the physical driver's actions or the platform's ability to assign a vehicle?
- APP_AND_MAPS: Does this evaluate digital software performance or geolocation accuracy?
- PRICING_AND_BILLING: Does this involve money, transactional mechanics, or fee perception?
- FULFILLMENT_FOOD: Does this evaluate the physical items inside the bag or the merchant's preparation workflow?

In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from src.preprocessing.transform import normalize_review, drop_duplicate_review
import config

# Using TF-IDF

In [9]:
df = pd.read_csv(config.RAW_DATA_PATH).pipe(normalize_review)
df

,userName,score,at,content
0,Pengguna Google,1,2026-09-03 09:16:13,barang ketinggalan udah minta bantu bantuan pu...
1,Pengguna Google,5,2026-09-03 09:16:06,drivernya ramah dan jalannya berhati2
2,Pengguna Google,1,2026-09-03 09:11:34,turunin harga grab
3,Pengguna Google,5,2026-09-03 08:57:58,good job
4,Pengguna Google,5,2026-09-03 08:32:12,sangat puas
...,...,...,...,...
9995,Hesti Aprilia,1,2026-04-15 21:04:13,makin kesini makin aneh udah 3x lebih order gr...
9996,Hearty Tandur,4,2026-04-15 21:02:52,banyakin diskon d ng
9997,Ades Arka,5,2026-04-15 20:52:16,mantap
9998,Lisa Aini,5,2026-04-15 20:51:24,pelayan sangat baik


In [136]:
TAXONOMY_STOP_WORDS = [
    # Conjunctions, prepositions, and structural fillers
    'yang', 'yg', 'dan', 'di', 'ke', 'dari', 'untuk', 'pada', 'atau',
    'oleh', 'dengan', 'dg', 'dgn', 'karena', 'karna', 'krn', 'sebab',
    'jika', 'kalau', 'kalo', 'kl', 'klo', 'maka', 'sehingga', 'supaya',
    'biar', 'ini', 'itu', 'tersebut', 'tentang', 'serta', 'sampai', 'sampe',

    # Pronouns & addresses
    'saya', 'sy', 'aku', 'gw', 'gue', 'gua', 'kami', 'kita',
    'anda', 'kamu', 'lu', 'lo', 'ente', 'mereka', 'dia', 'nya',

    # Conversational discourse particles & slang fillers
    'sih', 'deh', 'dong', 'kan', 'lah', 'pun', 'ya', 'yah',
    'nih', 'tuh', 'kok', 'loh', 'lho', 'kek', 'wkwk', 'wkwkwk', 'anjir',

    # Pure negations (stripped so nouns and root actions bind together)
    'ga', 'gak', 'gk', 'nggak', 'tidak', 'tdk', 'bukan', 'bkn', 
    'belum', 'blm', 'kurang', 'krg', 'tanpa',

    # Generic app/service boilerplate
    'aplikasi', 'apk', 'app', 'grab', 'fitur', 'sistem', 'tolong', 'mohon',
    'bikin', 'buat', 'kasih', 'jadi', 'jd', 'bisa', 'bs', 'udah',
    'udh', 'sudah', 'sdh', 'masih', 'msh', 'lagi', 'lg', 'selalu',
    'terus', 'trs', 'juga', 'jg', 'ada', 'ad', 'pernah', 'selesai',

    # Pure evaluative adjectives (positive & negative sentiment noise)
    'bagus', 'bgs', 'mantap', 'mantul', 'keren', 'oke', 'ok', 'top', 
    'terbaik', 'suka', 'puas', 'juara', 'membantu', 'nyaman', 'aman',
    'jelek', 'jlk', 'buruk', 'parah', 'kecewa', 'hancur', 'ancur', 
    'bobrok', 'payah', 'rugi', 'nyesel', 'rusak', 'benci', 'sampah',
    'good',

    'sangat', 'banget', 'tiba', 'sekali', 'terlalu', 'mau', 'makin'
]

In [144]:
tfidf = TfidfVectorizer(
    ngram_range = (2, 3),
    stop_words = TAXONOMY_STOP_WORDS
)

matrix = tfidf.fit_transform(df['content'])
phrases = tfidf.get_feature_names_out()

neg_mask  = (df['score'] <= 2).values
pos_mask = (df['score'] >= 4).values

neg_weights = np.asarray(matrix[neg_mask].mean(0)).ravel()
pos_weights = np.asarray(matrix[pos_mask].mean(0)).ravel()
weights = np.asarray(matrix.mean(0)).ravel()

results_df = pd.DataFrame({
    'term': phrases,
    'weights' : weights,
    'neg_mean_weight': neg_weights,
    'pos_mean_weight': pos_weights
})

print(results_df.sort_values(by = 'weights', ascending = False).head(40).reset_index(drop = True))
print('---')
print(results_df.sort_values(by = 'pos_mean_weight', ascending = False).head(40).reset_index(drop = True))
print('---')
print(results_df.sort_values(by = 'neg_mean_weight', ascending = False).head(40).reset_index(drop = True))

                     term   weights  neg_mean_weight  pos_mean_weight
0             tepat waktu  0.007719         0.000165         0.010860
1              baik ramah  0.004444         0.000000         0.006306
2            driver ramah  0.003423         0.000768         0.004556
3                the best  0.003038         0.000046         0.004294
4              luar biasa  0.002524         0.000094         0.003548
5              ramah baik  0.002310         0.000000         0.003278
6          pelayanan baik  0.001959         0.000000         0.002780
7             ramah sopan  0.001882         0.000178         0.002606
8              cukup baik  0.001675         0.000000         0.002306
9             cepat ramah  0.001657         0.000000         0.002352
10            cepat tepat  0.001546         0.000000         0.002194
11        pengemudi ramah  0.001459         0.000041         0.002055
12        drivernya ramah  0.001396         0.000069         0.001956
13            mudah 

| Operational Category | Negative Friction $n$-grams (`neg_mean_weight`) | Positive Praise $n$-grams (`pos_mean_weight`) |
| --- | --- | --- |
| **`DRIVER_OPERATIONS`** | `dapet driver`, `dapat driver`, `cari driver`, `driver lama`, `nyari driver`, `driver males`, `pengemudi lama`, `susah dapet`, `susah dapat`, `malah cancel`, `membatalkan pesanan`, `lama driver`, `banyak driver` | `driver ramah`, `drivernya ramah`, `pengemudi ramah`, `sopir ramah`, `driver baik`, `drivernya baik`, `pengemudi baik`, `baik ramah`, `ramah baik`, `ramah sopan`, `baik sopan`, `orangnya ramah`, `ramah cepat`, `tepat waktu` |
| **`APP_AND_MAPS`** | `banyak bug`, `sering eror`, `loading lama`, `kebanyakan iklan`, `update mulu`, `kebanyakan update`, `lokasi akurat` *(context: "tidak akurat")*, `sesuai titik` *(context: "tidak sesuai")* | `mudah digunakan`, `lebih mudah`, `mudah cepat`, `cepat mudah`, `sesuai titik` *(when accurate)* |
| **`PRICING_AND_BILLING`** | `ongkir mahal`, `ongkos mahal`, `tarif mahal`, `metode pembayaran`, `saldo ovo` | `banyak promo`, `lebih murah` |
| **`FULFILLMENT_FOOD`** | `grabfood lama`, `pesan grabfood`, `pesan makanan`, `pesen makanan`, `order makanan`, `pesen makan` | `pesan makanan`, `pesen makanan` *(general usage without friction)* |

---

### Non-Category Boilerplate & Unassigned Signals

* **Cross-Cutting Latency Symptoms:** `nunggu lama`, `lama nunggu`, `nunggu jam`, `jam lebih`, `lama lama`, `berkali kali` (Context-dependent; maps dynamically to Driver, Food Prep, or App freezes).
* **Generic Service Boilerplate (Positive):** `pelayanan baik`, `pelayanan ramah`, `pelayanan memuaskan`, `layanan baik`, `pelayanan cepat`, `respon cepat`, `memudahkan perjalanan`, `lebih baik`, `cukup baik`.
* **Superlative Sentiment Noise:** `the best`, `is the best`, `is the`, `luar biasa`.
* **Conversational Residuals:** `sama aja`, `sama driver`.

all fit nicely to the 4 category with no underrepresented class

# Manual Labelling on 50 reviews

In [5]:
df_processed = pd.read_csv(config.CLEANED_DATA_PATH)
composition = df_processed['score'].value_counts() / len(df_processed)

In [6]:
selected = df_processed.groupby('score').sample(frac = 50.2 / len(df_processed), random_state = 43)
selected.content

3493    aplikasi paling aneh bisa pilih makanan menu p...
3005    ga recommended mesen makan di sini suruh nonto...
2322    padahal udah nunggu makanannya tpi ternyata da...
2168    apk scam mau beli makanan harga awal 60k gara ...
3193    sistem pembayaran pakai ovo mlah bikin saldo n...
2734    woii grab tolong dong itu titik penjemputan di...
2122    grab food bener2 mengecewakan segala ad biaya ...
123     woii grab kalo ngasih petunjuk maps yang bener...
1848                   resto tutup tapi ga bisa di cancle
3036       grab tambah jelek sdh 3x terbatalkan tdk jelas
2424       apa sih tulisan qris malah qris bayar ditempat
2620    orderan ngelonjak pada saat hujan lama dapatny...
815     hallo saya dari salah satu penumpang grab bene...
1376    saya memesan gofood namun dari kurir tidak ada...
2399    apk sangat amat buruk sy sebagai pengguna sang...
2907    kenapa sih lelet banget ngelag jaringan normal...
3476    baru semalem di update belom di pake pagi nya ...
1109    apk ga

In [8]:
    df_processed['score'].value_counts()

score
1    1715
5    1175
2     307
3     293
4     183
Name: count, dtype: int64

In [15]:
df['score'].value_counts()

score
5    6570
1    2189
4     477
3     384
2     380
Name: count, dtype: int64